In [0]:
# Update the weather reference files after station-info silver completes.
# We will run this after the station_info updates to see if we need to start pulling new weather data.
# Probably we could get by with much fewer weather stations but whatever, it may still be nice to get it
#
# hrrr_locations.json
# model: weather model to request.
# grid_options: settings shared by mapping and forecast downloads.
# locations: location_id and representative latitude/longitude for each cell.
#
# station_weather_lookup.csv
# station_id: Citi Bike station identifier.
# lat, lon: observed station coordinates.
# location_id: assigned weather cell, matching the JSON location list.

# Our imports

from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import time
import pandas as pd

# Standard boilerplate
# Read production station information, with separate development output files.
try:
    RUN_MODE = dbutils.widgets.get("run_mode")
except Exception:
    RUN_MODE = "dev"

if RUN_MODE not in ("dev", "production"):
    raise ValueError("run_mode must be 'dev' or 'production'")

SOURCE_TABLE = "citibike_project.citibike.silver_station_info"
WEATHER_ROOT = Path("/Volumes/citibike_project/citibike/raw/weather_forecast")
REFERENCE_DIRECTORY = (
    WEATHER_ROOT / "reference"
    if RUN_MODE == "production" else WEATHER_ROOT / "_dev/reference"
)
API_URL = "https://api.open-meteo.com/v1/forecast"

# Load the initialized files, preserving coordinate precision from the CSV.
configuration_path = REFERENCE_DIRECTORY / "hrrr_locations.json"
lookup_path = REFERENCE_DIRECTORY / "station_weather_lookup.csv"

weather_configuration = json.loads(configuration_path.read_text())
station_weather_lookup = pd.read_csv(
    lookup_path,
    dtype={"station_id": "string", "location_id": "string"},
    float_precision="round_trip",
)

# Find station/coordinate pairs that are not already mapped.
station_locations = (
    spark.table(SOURCE_TABLE)
    .select("station_id", "lat", "lon")
    .distinct()
    .toPandas()
)
station_comparison = station_locations.merge(
    station_weather_lookup,
    on=["station_id", "lat", "lon"],
    how="left",
    validate="one_to_one",
)
new_stations = station_comparison.loc[
    station_comparison["location_id"].isna(),
    ["station_id", "lat", "lon"],
].copy()

# Just some paranoia

if len(new_stations) > 500:
    raise ValueError(
        f"{len(new_stations)} unmapped station locations; expected a handful. "
        "Check for a coordinate type or precision change before remapping."
    )

# Request the assigned weather cell for each new station location.
new_cell_ids = []

for batch_start in range(0, len(new_stations), 50):
    location_batch = new_stations.iloc[batch_start:batch_start + 50]

    request_parameters = {
        "latitude": ",".join(location_batch["lat"].astype(str)),
        "longitude": ",".join(location_batch["lon"].astype(str)),
        "models": weather_configuration["model"],
        **weather_configuration["grid_options"],
        "elevation": ",".join(
            [weather_configuration["grid_options"]["elevation"]] * len(location_batch)
        ),
    }
    with urlopen(
        f"{API_URL}?{urlencode(request_parameters)}", timeout=60
    ) as response:
        cells = json.load(response)

    if isinstance(cells, dict):
        cells = [cells]

    if len(cells) != len(location_batch):
        raise ValueError(
            f"Unexpected number of API results: expected "
            f"{len(location_batch)}, got {len(cells)}"
        )

    new_cell_ids.extend(
        f"hrrr_{cell['latitude']:.6f}_{cell['longitude']:.6f}"
        for cell in cells
    )
    time.sleep(6)

new_stations["location_id"] = new_cell_ids

if not new_stations.empty:
    # Append only weather cells that are not already in the configuration.
    existing_ids = {
        location["location_id"]
        for location in weather_configuration["locations"]
    }
    new_weather_locations = (
        new_stations.loc[~new_stations["location_id"].isin(existing_ids)]
        .drop_duplicates("location_id")
        .rename(columns={"lat": "latitude", "lon": "longitude"})
        [["location_id", "latitude", "longitude"]]
    )
    weather_configuration["locations"].extend(
        new_weather_locations.to_dict(orient="records")
    )

    # Append the new mappings while retaining all existing entries.
    station_weather_lookup = pd.concat(
        [station_weather_lookup, new_stations], ignore_index=True
    )

    # Save completed files before replacing the existing versions.
    configuration_temporary = configuration_path.with_suffix(".json.tmp")
    configuration_temporary.write_text(
        json.dumps(weather_configuration, indent=2), encoding="utf-8"
    )
    configuration_temporary.replace(configuration_path)

    lookup_temporary = lookup_path.with_suffix(".csv.tmp")
    station_weather_lookup.to_csv(lookup_temporary, index=False)
    lookup_temporary.replace(lookup_path)

    print(
        f"Added {len(new_stations)} station mappings and "
        f"{len(new_weather_locations)} weather cells."
    )
else:
    print("No new station locations; no requests or file changes.")